# Notebook de Testeo de Modelos de Regresion

Este notebook contiene 4 modelos separados por Markdown:

- CatBoost Regressor

Cada modelo incluye metricas y su importancia de variables.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib.util
import subprocess
import sys
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.set_option('display.max_columns',None)
sns.set(style="whitegrid")

In [ ]:
DATA_PATH = "../data/raw/kc_house_data.csv"
TARGET = "price"
RANDOM_STATE = 42
TEST_SIZE = 0.2

def evaluar_modelo(nombre, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{nombre}")
    print(f"MAE : {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R2  : {r2:.4f}")

def obtener_metricas(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

def normalizar_target(y_train, y_test):
    transformador = 'log1p'
    y_train_transformado = np.log1p(y_train.to_numpy())
    y_test_transformado = np.log1p(y_test.to_numpy())
    return transformador, y_train_transformado, y_test_transformado

def desnormalizar_target(transformador, valores):
    valores = np.asarray(valores)
    if transformador == 'log1p':
        return np.expm1(valores)
    return valores

def plot_importancia(df_importancia, titulo, top_n=15):
    top = df_importancia.sort_values("importance", ascending=False).head(top_n)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top, x="importance", y="feature", palette="Blues_r")
    plt.title(titulo)
    plt.xlabel("Importancia")
    plt.ylabel("Variable")
    plt.tight_layout()
    plt.show()


## Carga y Preparacion de Datos

In [ ]:
data_modelo = pd.read_csv(DATA_PATH)
data_modelo['zipcode']=data_modelo['zipcode'].astype('category')
anio_actual = pd.Timestamp.today().year  

data_modelo['data_modelod_fabricacion'] = anio_actual - data_modelo['yr_built']
data_modelo['data_modelod_renovacion'] = np.where(data_modelo['yr_renovated'] > 0, anio_actual - data_modelo['yr_renovated'], 0)
if "date" in data_modelo.columns:
    data_modelo["date"] = pd.to_datetime(data_modelo["date"], format="%Y%m%dT%H%M%S", errors="coerce")
    data_modelo["sale_year"] = data_modelo["date"].dt.year
    data_modelo["sale_month"] = data_modelo["date"].dt.month
    data_modelo["sale_day"] = data_modelo["date"].dt.day

data_modelo = data_modelo.drop(columns=["id", "date"], errors="ignore")


In [ ]:

data_modelo=data_modelo.drop(columns=['yr_built', 'yr_renovated','sale_year', 'sale_month', 'sale_day','zipcode','sqft_living15', 'sqft_lot15'])

## Variables categoricas 

## Variables completamente numericas

## 3) Transicion a una funcion reusable

El entrenamiento del modelo se encapsula en una funcion para evitar repetir bloques manuales en el notebook.


## 4) Funcion reutilizable para CatBoost

En lugar de dejar bloques separados de entrenamiento, comparacion y `grid search`, se define una sola funcion que:
- prepara el dataset desde el CSV original
- genera las variables derivadas de fechas
- separa ordinales y categoricas nominales
- elimina columnas opcionales
- aplica transformacion logaritmica al target si se solicita
- entrena el modelo base y, si se desea, ejecuta `grid_search`


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("catboost") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "catboost"])

from catboost import CatBoostRegressor, Pool


In [ ]:
def ejecutar_catboost_regresion(
    df,
    target,
    columnas_ordinales=None,
    columnas_categoricas_nominales=None,
    columnas_eliminar=None,
    columnas_base_a_eliminar=None,
    params_modelo=None,
    param_grid=None,
    usar_target_logaritmico=True,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
):
    columnas_ordinales = columnas_ordinales or []
    columnas_categoricas_nominales = columnas_categoricas_nominales or []
    columnas_eliminar = columnas_eliminar or []
    columnas_base_a_eliminar = columnas_base_a_eliminar or [
        'id', 'date', 'yr_built', 'yr_renovated', 'sqft_living15', 'sqft_lot15'
    ]
    params_modelo = params_modelo or {
        'iterations': 600,
        'learning_rate': 0.05,
        'depth': 6,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': random_state,
        'verbose': 0,
        'allow_writing_files': False,
    }

    data_modelo = df.copy()
    anio_actual = pd.Timestamp.today().year

    if 'yr_built' in data_modelo.columns:
        data_modelo['data_modelod_fabricacion'] = anio_actual - data_modelo['yr_built']
    if 'yr_renovated' in data_modelo.columns:
        data_modelo['data_modelod_renovacion'] = np.where(
            data_modelo['yr_renovated'] > 0,
            anio_actual - data_modelo['yr_renovated'],
            0,
        )
        data_modelo['tiene_renov'] = np.where(
            data_modelo['data_modelod_renovacion'] == 0,
            'No',
            'Si',
        )

    data_modelo = data_modelo.drop(columns=columnas_base_a_eliminar, errors='ignore')
    data_modelo = data_modelo.drop(columns=columnas_eliminar, errors='ignore')

    cat_features = [
        col for col in columnas_ordinales + columnas_categoricas_nominales
        if col in data_modelo.columns and col != target
    ]
    for col in cat_features:
        data_modelo[col] = data_modelo[col].astype(str)

    X = data_modelo.drop(columns=[target])
    y = data_modelo[target].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    target_scaler = None
    y_train_modelo = y_train.copy()
    y_test_modelo = y_test.copy()

    if usar_target_logaritmico:
        target_scaler, y_train_modelo, y_test_modelo = normalizar_target(y_train, y_test)

    modelo = CatBoostRegressor(**params_modelo)
    modelo.fit(
        X_train,
        y_train_modelo,
        cat_features=cat_features,
        eval_set=(X_test, y_test_modelo),
        use_best_model=True,
    )

    y_pred_modelo = modelo.predict(X_test)
    y_pred = desnormalizar_target(target_scaler, y_pred_modelo) if target_scaler else y_pred_modelo

    metricas_base = pd.DataFrame(
        [{'modelo': 'base', **obtener_metricas(y_test, y_pred)}]
    ).set_index('modelo')

    importancia = pd.DataFrame(
        {
            'feature': X_train.columns,
            'importance': modelo.get_feature_importance(),
        }
    ).sort_values('importance', ascending=False)

    resultado = {
        'modelo': modelo,
        'metricas': metricas_base,
        'importancia': importancia,
        'predicciones': pd.DataFrame({'y_real': y_test, 'y_pred': y_pred}, index=y_test.index),
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'cat_features': cat_features,
        'target_logaritmico': usar_target_logaritmico,
        'grid': None,
    }

    if param_grid:
        params_grid_base = {k: v for k, v in params_modelo.items() if k not in param_grid}
        modelo_grid = CatBoostRegressor(**params_grid_base)
        train_pool = Pool(X_train, y_train_modelo, cat_features=cat_features)
        resultado_grid = modelo_grid.grid_search(
            param_grid,
            X=train_pool,
            search_by_train_test_split=True,
            train_size=0.8,
            partition_random_seed=random_state,
            calc_cv_statistics=True,
            refit=True,
            shuffle=True,
            stratified=False,
            verbose=False,
            plot=False,
        )

        y_pred_grid_modelo = modelo_grid.predict(X_test)
        y_pred_grid = desnormalizar_target(target_scaler, y_pred_grid_modelo) if target_scaler else y_pred_grid_modelo

        resumen_grid = pd.DataFrame(
            [
                {'modelo': 'base', **obtener_metricas(y_test, y_pred)},
                {'modelo': 'grid_search', **obtener_metricas(y_test, y_pred_grid)},
            ]
        ).set_index('modelo')

        resultado['grid'] = {
            'modelo': modelo_grid,
            'mejores_parametros': resultado_grid['params'],
            'resumen_metricas': resumen_grid,
            'predicciones': pd.DataFrame({'y_real': y_test, 'y_pred': y_pred_grid}, index=y_test.index),
            'resultado_bruto': resultado_grid,
        }

    return resultado


In [ ]:
catboost_params = {
    'iterations': 1200,
    'learning_rate': 0.03,
    'depth': 6,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'allow_writing_files': False,
}

param_grid_catboost = {
    'depth': [4, 6],
    'learning_rate': [0.05],
    'l2_leaf_reg': [3],
    'iterations': [100],
}

resultado_catboost = ejecutar_catboost_regresion(
    df=pd.read_csv(DATA_PATH),
    target=TARGET,
    columnas_ordinales=['view', 'grade'],
    columnas_categoricas_nominales=['zipcode'],
    columnas_eliminar=[
        'waterfront', 'data_modelod_fabricacion', 'sqft_basement', 'bathrooms',
        'condition', 'floors', 'bedrooms', 'tiene_renov', 'data_modelod_renovacion'
    ],
    params_modelo=catboost_params,
    param_grid=None,
    usar_target_logaritmico=True,
)

resultado_catboost['metricas']


In [ ]:
resultado_catboost['importancia'].head(10)


In [ ]:
plot_importancia(resultado_catboost['importancia'], 'Top Variables - CatBoost', top_n=15)


## 5) Ejemplo de salida del grid search

Si `param_grid` se envia a la funcion, el resultado queda disponible en `resultado_catboost['grid']`.


In [ ]:
if resultado_catboost['grid'] is not None:
    print('Mejores parametros encontrados:')
    print(resultado_catboost['grid']['mejores_parametros'])
    display(resultado_catboost['grid']['resumen_metricas'])
else:
    print('Grid search desactivado: se esta usando solo el modelo base seleccionado.')
    display(resultado_catboost['metricas'])
